# Phase 2.2: Extract Semantic Features using BERT

This notebook will download the Pandora Personality dataset, clean it for BERT, extract the `[CLS]` embeddings using `bert-base-uncased` on GPU, and save the embeddings to `.npy` files which you can download.

In [ ]:
!pip install datasets transformers pandas numpy tqdm torch

In [ ]:
import re
import numpy as np
import pandas as pd
import torch
from transformers import BertTokenizer, BertModel
from datasets import load_dataset
from tqdm.auto import tqdm
from google.colab import files

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')
model.to(device)
model.eval()

In [ ]:
print("Downloading Pandora Dataset...")
dataset = load_dataset('marchmallow/Automated-Personality-Prediction')

def clean_for_bert(text):
    if not isinstance(text, str):
        return ""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    return text.strip()

splits = ['train', 'validation', 'test']

for split in splits:
    print(f"\nProcessing {split} split...")
    df = dataset[split].to_pandas()
    texts = df['text'].apply(clean_for_bert).tolist()
    
    batch_size = 64
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size)):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(batch_texts, return_tensors='pt', padding=True, truncation=True, max_length=512)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            # Extract [CLS] token
            cls_emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            all_embeddings.append(cls_emb)
            
    if all_embeddings:
        emb_matrix = np.vstack(all_embeddings)
        filename = f"{split}_bert_embeddings.npy"
        np.save(filename, emb_matrix)
        print(f"Saved {filename} with shape {emb_matrix.shape}")
        files.download(filename) # Automatically download to local machine